# comfio LLM-Native Integration Tutorial

This notebook demonstrates how to use comfio as a **deterministic tool layer** for LLM agents — semantic serialization, function-calling schemas, and prompt templates.

| Section | Topic | Extra Required |
|---------|-------|---------------|
| 1 | Semantic interpreters (markdown + summary dict) | — |
| 2 | Markdown summary from raw sensor DataFrame | — |
| 3 | Prompt templates for edge deployment | — |
| 4 | Pydantic tool schemas (function calling) | `[agent]` |
| 5 | OpenAI tool definitions | `[agent]` |
| 6 | LangChain adapter | `[agent]` + langchain |

**Install**: `pip install comfio[agent]` for sections 4–6. Sections 1–3 work with core only.

In [ ]:
import numpy as np
import pandas as pd
import json
import traceback

from comfio import (
    evaluate_thermal,
    evaluate_visual,
    evaluate_acoustic,
    evaluate_iaq,
    calculate_global_ieq,
    calculate_compliance,
    ieq_to_markdown,
    ieq_to_summary_dict,
    generate_markdown_summary,
    EDGE_SYSTEM_PROMPT,
    DIAGNOSTIC_PROMPT_TEMPLATE,
    format_prompt,
)

# Simulate 24 hours of sensor data
rng = np.random.default_rng(42)
n = 24
df = pd.DataFrame({
    "temperature": 23 + rng.normal(0, 1.5, n),
    "radiant_temp": 23 + rng.normal(0, 1.5, n),
    "rh": 50 + rng.normal(0, 5, n),
    "air_speed": 0.1 + rng.normal(0, 0.02, n),
    "lux": 450 + rng.normal(0, 80, n),
    "laeq": 45 + rng.normal(0, 6, n),
    "co2": 700 + rng.normal(0, 120, n),
})

# Evaluate all domains
thermal = evaluate_thermal(
    tdb=df["temperature"].values, tr=df["radiant_temp"].values,
    vr=df["air_speed"].values, rh=df["rh"].values,
    met=1.2, clo=0.5, category="B",
)
visual = evaluate_visual(illuminance=df["lux"].values, task_type="office_writing")
acoustic = evaluate_acoustic(laeq=df["laeq"].values, nc_level="NC-35")
iaq = evaluate_iaq(co2=df["co2"].values, threshold_level="good")

ieq_result = calculate_global_ieq(
    thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq,
)
report = calculate_compliance(ieq_result, threshold=70.0)

print(f"IEQ Index avg: {np.mean(ieq_result.index):.1f}")
print(f"Compliance: {report.compliance_rate_pct:.1f}%")

## 1. Semantic Interpreters

`ieq_to_markdown()` converts a `GlobalIEQResult` into token-efficient structured markdown (~200 tokens vs 14,400 for raw 24h data). `ieq_to_summary_dict()` returns a flat dict for structured JSON injection.

In [ ]:
# Markdown output (for LLM context injection)
md = ieq_to_markdown(ieq_result, compliance_report=report, zone_id="Room-402")
print(md)
print(f"\n--- Token estimate: ~{len(md) // 4} tokens ({len(md)} chars) ---")

In [ ]:
# Summary dict (for structured JSON context)
summary = ieq_to_summary_dict(ieq_result, compliance_report=report)
print("Summary dict (for JSON injection):")
print(json.dumps(summary, indent=2, default=str))

## 2. Markdown Summary from Raw DataFrame

`generate_markdown_summary()` runs the full IEQ pipeline on a sensor DataFrame and produces a dense report — the foundation for RAG ingestion or historical analysis.

In [ ]:
# Generate full markdown report from raw sensor DataFrame
report_md = generate_markdown_summary(df, threshold=70.0, zone_id="Room-402")
print(report_md)

## 3. Prompt Templates for Edge Deployment

Pre-built system prompts with physics-guard rules for deploying comfio alongside local LLMs (Llama, Mistral) on edge gateways.

In [ ]:
# Edge deployment system prompt with IEQ context injected
prompt = format_prompt(
    EDGE_SYSTEM_PROMPT,
    context=md,
    query="Occupants in Room-402 complain it's warm and stuffy. Diagnose and recommend actions.",
)
print(prompt)

In [ ]:
# Diagnostic prompt template
diag_prompt = format_prompt(
    DIAGNOSTIC_PROMPT_TEMPLATE,
    ieq_report=md,
    complaint="Occupants feel drowsy and warm during afternoon hours.",
)
print(diag_prompt)

## 4. Pydantic Tool Schemas (Function Calling)

`evaluate_thermal_tool()` and `evaluate_ieq_tool()` wrap comfio's core calculations into LLM-callable functions with typed inputs (list[float]) and structured dict outputs. Requires `pip install comfio[agent]`.

In [ ]:
def try_run(label, fn):
    """Run a function, catching ImportError for missing extras."""
    try:
        result = fn()
        print(f"  [OK] {label}")
        return result
    except ImportError as e:
        print(f"  [SKIP] {label}")
        print(f"        {e}")
        return None
    except Exception:
        print(f"  [ERROR] {label}")
        traceback.print_exc()
        return None

def demo_thermal_tool():
    from comfio.llm.tools import evaluate_thermal_tool

    result = evaluate_thermal_tool(
        air_temperature=[23.0, 24.0, 25.0, 26.0],
        relative_humidity=[50.0, 55.0, 60.0, 58.0],
    )
    print("evaluate_thermal_tool result:")
    print(json.dumps(result, indent=2))
    return result

thermal_tool_result = try_run("evaluate_thermal_tool", demo_thermal_tool)

In [ ]:
def demo_ieq_tool():
    from comfio.llm.tools import evaluate_ieq_tool

    result = evaluate_ieq_tool(
        air_temperature=[23.0, 24.0, 25.0, 26.0],
        relative_humidity=[50.0, 55.0, 60.0, 58.0],
        illuminance=[500.0, 450.0, 480.0, 520.0],
        noise_laeq=[40.0, 42.0, 38.0, 45.0],
        co2=[800.0, 700.0, 750.0, 900.0],
    )
    print("evaluate_ieq_tool result:")
    # Print without markdown for brevity
    result_copy = {k: v for k, v in result.items() if k != "markdown"}
    print(json.dumps(result_copy, indent=2, default=str))
    print(f"\n--- Markdown ({len(result['markdown'])} chars) ---")
    print(result["markdown"])
    return result

ieq_tool_result = try_run("evaluate_ieq_tool", demo_ieq_tool)

## 5. OpenAI Tool Definitions

`to_openai_tools()` generates JSON schema dicts compatible with OpenAI's function calling API.

In [ ]:
def demo_openai_tools():
    from comfio.llm.tools import to_openai_tools

    tools = to_openai_tools()
    print(f"OpenAI tool definitions ({len(tools)} tools):\n")
    for t in tools:
        print(f"  {t['function']['name']}: {t['function']['description'][:80]}...")
    print(f"\nFull schema for evaluate_thermal:")
    print(json.dumps(tools[0], indent=2))
    return tools

openai_tools_result = try_run("to_openai_tools", demo_openai_tools)

## 6. LangChain Adapter

`to_langchain_tools()` returns `StructuredTool` objects ready for LangChain agents. Requires `pip install langchain`.

In [ ]:
def demo_langchain():
    from comfio.llm.tools import to_langchain_tools

    tools = to_langchain_tools()
    print(f"LangChain tools ({len(tools)}):")
    for t in tools:
        print(f"  {t.name}: {t.description[:80]}...")
    return tools

langchain_result = try_run("to_langchain_tools", demo_langchain)

## Summary

### No extra dependencies (core only)
- `ieq_to_markdown()` — token-efficient markdown from `GlobalIEQResult` (~200 tokens)
- `ieq_to_summary_dict()` — flat dict for structured JSON context
- `generate_markdown_summary()` — full IEQ pipeline on DataFrame → markdown report
- `EDGE_SYSTEM_PROMPT` / `DIAGNOSTIC_PROMPT_TEMPLATE` — guarded prompt templates
- `format_prompt()` — simple template formatter

### Requires `pip install comfio[agent]`
- `evaluate_thermal_tool()` — LLM-callable thermal comfort evaluation
- `evaluate_ieq_tool()` — LLM-callable multi-domain IEQ evaluation
- `to_openai_tools()` — JSON schemas for OpenAI function calling
- `to_langchain_tools()` — `StructuredTool` objects for LangChain agents

### Architecture
```
src/comfio/llm/
├── interpreters.py   # No extra deps
├── prompts.py        # No extra deps
└── tools.py          # Requires pydantic (optional [agent] extra)
```

Core physics modules in `domains/` remain completely isolated from LLM logic.

# comfio LLM-Native Integration Tutorial

This notebook demonstrates how to use comfio as a **deterministic tool layer** for LLM agents — semantic serialization, function-calling schemas, and prompt templates.

| Section | Topic | Extra Required |
|---------|-------|---------------|
| 1 | Semantic interpreters (markdown + summary dict) | — |
| 2 | Markdown summary from raw sensor DataFrame | — |
| 3 | Prompt templates for edge deployment | — |
| 4 | Pydantic tool schemas (function calling) | `[agent]` |
| 5 | OpenAI tool definitions | `[agent]` |
| 6 | LangChain adapter | `[agent]` + langchain |

**Install**: `pip install comfio[agent]` for sections 4–6. Sections 1–3 work with core only.